# Chapter 13: silver_vessel_event and a pipeline you can run twice

Section 13.3 builds silver_vessel_event for the Kestrel slice and prints the quarantine table. Section 13.5
runs the slice twice and proves the second run changes no rows. In the slice only FV07 carries a historian
trace, so the detected events are FV07's; the other fills come from the typed brew sheet and transfer log.

In [ ]:
import sys
sys.path.insert(0, "..")
import sqlite3
import tempfile
from pathlib import Path
import pandas as pd
import kestrel as k
import pipeline

tags = k.load("tag_reading")
tag_map = k.load("tag_map")
silver_tag_reading = tags.merge(tag_map[tag_map.valid_to.isna()][["tag", "vessel", "measurement", "unit"]],
                                on="tag", how="left")
print(silver_tag_reading.groupby(["tag", "measurement"]).size().to_string())

In [ ]:
silver_vessel_event = k.match_events(k.detect_events(k.wide_trace(tags, "FV07"), "FV07"))
print(silver_vessel_event.to_string(index=False))

Rows that fail a rule go to silver_quarantine with a reason. Here: the event with no match, plus the five
checks of chapter 12.

In [ ]:
no_match = silver_vessel_event[silver_vessel_event.matched_to.isna()]
quarantine = pd.concat([
    pd.DataFrame({"row_ref": "event:FV07:" + no_match.event_type + ":" + no_match.start_utc.astype(str),
                  "check_failed": "join", "reason_code": "NO_MATCH",
                  "date": no_match.start_utc.astype(str).str[:10], "owner": "Head brewer"}),
    k.run_checks(k.connect()),
], ignore_index=True)
print(quarantine.to_string(index=False))

The companion pipeline: bronze, silver and gold in one SQLite file, run twice from a clean start.

In [ ]:
db = Path(tempfile.mkdtemp()) / "kestrel.db"
first = pipeline.run(db)
second = pipeline.run(db)
for name, r in [("first", first), ("second", second)]:
    print(f"{name} run: {r['files_landed']} files landed, {r['silver_gold_rows_changed']} silver and gold rows written")
assert second["silver_gold_rows_changed"] == 0 and first["fingerprint"] == second["fingerprint"]
print(f"the second run changed no rows in {len(second['fingerprint'])} silver and gold tables")
con = sqlite3.connect(db)
print(pd.read_sql_query("SELECT * FROM silver_quarantine", con).to_string(index=False))